# Preprocessing: None values only

In [1]:
import pandas as pd

files = [
    r'..\Data-Working\\BAJAJ-Historical-Data.csv', 
    r'..\Data-Working\\HDFCBank-Historical-Data.csv', 
    r'..\Data-Working\\Hindalco-Historical-Data.csv',
    r'..\Data-Working\\HUL-Historical-Data.csv', 
    r'..\Data-Working\\ICICIBank-Historical-Data.csv', 
    r'..\Data-Working\\Infosys-Historical-Data.csv',
    r'..\Data-Working\\ITC-Historical-Data.csv', 
    r'..\Data-Working\\L&T-Historical-Data.csv', 
    r'..\Data-Working\\Nifty-50-Historical-Data.csv',
    r'..\Data-Working\\RelianceIndustries-Historical-Data.csv', 
    r'..\Data-Working\\SBI-Historical-Data.csv', 
    r'..\Data-Working\\TATAMotors-Historical-Data.csv',
    r'..\Data-Working\\TataSteel-Historical-Data.csv', 
    r'..\Data-Working\\TCS-Historical-Data.csv'
]

print("Missing Values Summary:")
print("=" * 50)

for file in files:
    try:
        data = pd.read_csv(file, index_col=0, parse_dates=True)
        missing_values = data.isna().sum()
        total_missing = missing_values.sum()
        
        # Extract stock name from filename
        stock_name = file.split('\\')[-1].split('-Historical')[0]
        
        print(f"{stock_name:<20}: {total_missing} missing values")
        
        # Show detailed breakdown if there are missing values
        if total_missing > 0:
            for col, missing_count in missing_values[missing_values > 0].items():
                print(f"  - {col}: {missing_count}")
                
    except FileNotFoundError:
        print(f"File not found: {file}")
    except Exception as e:
        print(f"Error processing {file}: {e}")

print("=" * 50)

# data = data.fillna(method='ffill')  # forward fill
# data = data.dropna()  # just in case

Missing Values Summary:
BAJAJ               : 0 missing values
HDFCBank            : 0 missing values
Hindalco            : 0 missing values
HUL                 : 0 missing values
ICICIBank           : 0 missing values
Infosys             : 0 missing values
ITC                 : 0 missing values
L&T                 : 0 missing values
Nifty-50            : 0 missing values
RelianceIndustries  : 0 missing values
SBI                 : 0 missing values
TATAMotors          : 0 missing values
TataSteel           : 0 missing values
TCS                 : 0 missing values


# Log Returns for each stock

In [2]:
import pandas as pd
import numpy as np


for file in files:
    df = pd.read_csv(file)
    print(f"Processing {file}")
    print(f"Data types: {df.dtypes}")
    # print(f"Price column type: {type(df['Price'].iloc[0])}")


Processing ..\Data-Working\\BAJAJ-Historical-Data.csv
Data types: Date        object
Price       object
Open        object
High        object
Low         object
Vol.        object
Change %    object
dtype: object
Processing ..\Data-Working\\HDFCBank-Historical-Data.csv
Data types: Date        object
Price       object
Open        object
High        object
Low         object
Vol.        object
Change %    object
dtype: object
Processing ..\Data-Working\\Hindalco-Historical-Data.csv
Data types: Date         object
Price       float64
Open        float64
High        float64
Low         float64
Vol.         object
Change %     object
dtype: object
Processing ..\Data-Working\\HUL-Historical-Data.csv
Data types: Date        object
Price       object
Open        object
High        object
Low         object
Vol.        object
Change %    object
dtype: object
Processing ..\Data-Working\\ICICIBank-Historical-Data.csv
Data types: Date        object
Price       object
Open        object
High      

In [3]:
for file in files:
    try:
        df = pd.read_csv(file)
        stock_name = file.split('\\')[-1].split('-Historical')[0]
        print(f"Processing {stock_name}")
        
        # Handle Price column
        if df['Price'].dtype == 'object':
            df['Price'] = df['Price'].astype(str).str.replace(',', '').astype(float)
        
        # Process data
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.sort_values('Date').reset_index(drop=True)
        df['Log-Return'] = np.log(df['Price'] / df['Price'].shift(1))
        df = df.dropna(subset=['Log-Return'])
        
        # ✅ Save to new file with "_processed" suffix
        processed_file = file.replace('.csv', '_with_returns.csv')
        df.to_csv(processed_file, index=False)
        
        print(f"  Rows: {len(df)}")
        print(f"  Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
        print(f"  Log-Return column added successfully!")
        print(f"  ✓ Saved as: {processed_file.split('\\')[-1]}\n")
        
    except Exception as e:
        print(f"  ✗ Error: {e}\n")
        continue

Processing BAJAJ
  Rows: 2664
  Date range: 2015-01-02 to 2025-10-03
  Log-Return column added successfully!
  ✓ Saved as: BAJAJ-Historical-Data_with_returns.csv

Processing HDFCBank
  Rows: 2665
  Date range: 2015-01-02 to 2025-10-03
  Log-Return column added successfully!
  ✓ Saved as: HDFCBank-Historical-Data_with_returns.csv

Processing Hindalco
  Rows: 2664
  Date range: 2015-01-02 to 2025-10-03
  Log-Return column added successfully!
  ✓ Saved as: Hindalco-Historical-Data_with_returns.csv

Processing HUL
  Rows: 2664
  Date range: 2015-01-02 to 2025-10-03
  Log-Return column added successfully!
  ✓ Saved as: HUL-Historical-Data_with_returns.csv

Processing ICICIBank
  Rows: 2664
  Date range: 2015-01-02 to 2025-10-03
  Log-Return column added successfully!
  ✓ Saved as: ICICIBank-Historical-Data_with_returns.csv

Processing Infosys
  Rows: 2664
  Date range: 2015-01-02 to 2025-10-03
  Log-Return column added successfully!
  ✓ Saved as: Infosys-Historical-Data_with_returns.csv

Pr

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

def calibrate_gbm_annualized(log_returns, dt):
    """
    Calibrate ANNUALIZED GBM parameters using MLE.
    This uses the correct logic from your first script.
    """
    r = log_returns.values
    
    # Calculate annualized drift (mu)
    mu_hat = np.mean(r)/dt + 0.5*np.var(r)/dt
    
    # Calculate annualized volatility (sigma)
    sigma_hat = np.std(r)/np.sqrt(dt)
    
    return mu_hat, sigma_hat

def simulate_gbm(S0, mu, sigma, T, dt, n_paths=1000):
    """
    Simulate GBM paths using ANNUALIZED parameters.
    This function is good as-is.
    """
    n_steps = int(T/dt)
    t = np.linspace(0, T, n_steps)
    S = np.zeros((n_paths, n_steps))
    S[:, 0] = S0
    
    for i in range(1, n_steps):
        Z = np.random.standard_normal(n_paths)
        # This formula correctly uses annualized mu/sigma with fractional dt
        S[:, i] = S[:, i-1] * np.exp((mu - 0.5*sigma**2)*dt + sigma*np.sqrt(dt)*Z)
    
    return t, S  # Return both time and paths

# Global parameters
dt = 1/252
T = 1
n_paths_to_sim = 1000

for file in files:
    try:
        df = pd.read_csv(file)
        stock_name = file.split('\\')[-1].split('-Historical')[0]
        print(f"Processing {stock_name}")
        
        # 1. Clean data (same as your script)
        if df['Price'].dtype == 'object':
            df['Price'] = df['Price'].astype(str).str.replace(',', '').astype(float)
        df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
        df = df.sort_values('Date').reset_index(drop=True)
        df['Log-Return'] = np.log(df['Price'] / df['Price'].shift(1))
        df = df.dropna(subset=['Log-Return'])
        
        # 2. Calibrate (using your new, correct function)
        mu, sigma = calibrate_gbm_annualized(df['Log-Return'], dt)
        print(f"  Annual μ (drift): {mu:.5f}")
        print(f"  Annual σ (volatility): {sigma:.5f}")

        # 3. Simulate (using your new, correct function)
        S0 = df['Price'].iloc[-1]
        time_array, sim_paths = simulate_gbm(S0, mu, sigma, T, dt, n_paths_to_sim)
        
        # 4. Plot (optional)
        plt.figure(figsize=(10, 5))
        plt.plot(time_array, sim_paths.T, alpha=0.1, color='blue') # .T transposes for plotting
        plt.plot(time_array, np.mean(sim_paths, axis=0), color='red', linewidth=2) # Plot mean path
        plt.title(f"{n_paths_to_sim} Simulated GBM Paths — {stock_name}")
        plt.xlabel("Time (Years)")
        plt.ylabel("Price")
        plt.show()

        # 5. DO YOUR "OTHER STUFFS" HERE
        # Now you have the raw simulation data in 'sim_paths'
        
        # Example: Calculate expected price and 95% CI at end of year
        final_prices = sim_paths[:, -1]
        expected_price = np.mean(final_prices)
        percentile_5 = np.percentile(final_prices, 5)
        percentile_95 = np.percentile(final_prices, 95)
        
        print(f"  Last Known Price: {S0:.2f}")
        print(f"  Expected Price in 1 Year: {expected_price:.2f}")
        print(f"  95% C.I. for Final Price: ({percentile_5:.2f} - {percentile_95:.2f})\n")
        
    except Exception as e:
        print(f"  ✗ Error processing {file}: {e}\n")
        continue

In [ ]:
import pandas as pd
import numpy as np
import os

# --- Helpers ---
def parse_volume_strict(vol_str):
    """Convert 'K', 'M', 'B' suffixed volumes to float; return NaN if invalid."""
    if isinstance(vol_str, str):
        vol_str = vol_str.strip().replace(',', '')
        try:
            if vol_str.endswith('K'):
                return float(vol_str[:-1]) * 1_000
            elif vol_str.endswith('M'):
                return float(vol_str[:-1]) * 1_000_000
            elif vol_str.endswith('B'):
                return float(vol_str[:-1]) * 1_000_000_000
            else:
                return float(vol_str)
        except ValueError:
            return np.nan # Return NaN on failure instead of raising error
    return np.nan # Return NaN if not a string or other issue

def parse_change_strict(change_str):
    """Remove % and convert to float; return NaN if invalid."""
    if isinstance(change_str, str):
        change_str = change_str.strip().replace('%', '').replace(',', '')
        try:
            return float(change_str)
        except ValueError:
            return np.nan # Return NaN on failure
    return np.nan # Return NaN if not a string or other issue

def clean_fill_and_save_inplace(file_path):
    """
    Clean a single CSV, fill missing values, and save it in-place.
    """
    print(f"\nProcessing file: {file_path}")
    df = pd.read_csv(file_path)

    # --- Date Parsing ---
    # Try to parse dates, but don't fail; bad dates will be NaT
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

    # --- Column Conversions ---
    # Convert numeric columns, forcing errors to NaN
    numeric_cols = ['Price', 'Open', 'High', 'Low']
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(
                df[col].astype(str).str.replace(',', '', regex=False), 
                errors='coerce'
            )

    # Parse Volume, forcing errors to NaN
    if 'Vol.' in df.columns:
        df['Vol.'] = df['Vol.'].apply(parse_volume_strict)

    # Parse Change %, forcing errors to NaN
    if 'Change %' in df.columns:
        df['Change %'] = df['Change %'].apply(parse_change_strict)

    # --- Sort by Date ---
    # Must sort by date BEFORE filling
    df = df.sort_values('Date').reset_index(drop=True)

    # --- Fill Missing Values ---
    # 1. Forward-fill: Fills NaN with the value from the previous day
    df.ffill(inplace=True)
    # 2. Backward-fill: Fills any remaining NaNs (e.g., at the start)
    df.bfill(inplace=True)

    # Final check: if the *whole* file was bad, df might still have NaNs
    if df.isnull().values.any():
        raise ValueError(f"File {file_path} still contains NaNs after filling.")

    # --- Save In-Place ---
    # Overwrite the original file
    df.to_csv(file_path, index=False)
    
    print(f"✅ Cleaned, filled, and saved: {file_path}")
    return 


# Loop and process each file
for file in files:
    try:
        clean_fill_and_save_inplace(file)
    except Exception as e:
        print(f"  ✗ FAILED to process {file}: {e}\n")
        continue  # Skip to the next file

print("\n--- All files processed. ---")

In [ ]:
import pandas as pd
import numpy as np


for file in files:
    df = pd.read_csv(file)
    print(f"Processing {file}")
    print(f"Data types:\n {df.dtypes}")

In [ ]:
import pandas as pd
import numpy as np
import os

# --- Helper to parse 'K', 'M', 'B' ---
def parse_volume_to_float(vol_str):
    """Convert 'K', 'M', 'B' suffixed volumes to float; return NaN if invalid."""
    if isinstance(vol_str, str):
        vol_str = vol_str.strip().replace(',', '')
        try:
            if vol_str.endswith('K'):
                return float(vol_str[:-1]) * 1_000
            elif vol_str.endswith('M'):
                return float(vol_str[:-1]) * 1_000_000
            elif vol_str.endswith('B'):
                return float(vol_str[:-1]) * 1_000_000_000
            else:
                return float(vol_str)
        except ValueError:
            return np.nan # Return NaN on failure
    
    # Handle if it's already a float/int
    elif isinstance(vol_str, (int, float)):
        return float(vol_str)
        
    return np.nan # Return NaN if not a string or other issue

def convert_volume_to_int_inplace(file_path):
    """
    Reads a CSV, converts its 'Vol.' column to int, and saves in-place.
    """
    print(f"\nProcessing file: {file_path}")
    df = pd.read_csv(file_path)

    if 'Vol.' not in df.columns:
        print(f"  Warning: 'Vol.' column not found in {file_path}. Skipping.")
        return

    # 1. Parse 'K', 'M', 'B' strings to float
    df['Vol.'] = df['Vol.'].apply(parse_volume_to_float)
    
    # 2. Fill missing values
    #    We must fill NaNs before we can convert to integer
    df['Vol.'].ffill(inplace=True) # Forward-fill
    df['Vol.'].bfill(inplace=True) # Backward-fill (for any at the start)

    # 3. Round and Convert to Integer
    df['Vol.'] = df['Vol.'].round().astype(np.int64)

    # 4. Save In-Place
    #    Overwrite the original file
    df.to_csv(file_path, index=False)
    
    print(f"✅ 'Vol.' column converted to int and saved: {file_path}")
    return


# --- Main Logic ---
folder_path = r"..\Data-Working"  # Your data folder
files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) 
         if f.endswith('.csv') and '_copy' not in f]

print(f"Found {len(files)} files to process in-place...")

# Loop and process each file
for file in files:
    try:
        convert_volume_to_int_inplace(file)
    except Exception as e:
        print(f"  ✗ FAILED to process {file}: {e}\n")
        continue  # Skip to the next file

print("\n--- All files processed. ---")